In [ ]:
import warnings
warnings.filterwarnings('ignore')
# output: no output — warnings hidden

In [ ]:
import micropip
await micropip.install("imbalanced-learn")
# output: installs imbalanced-learn, shows progress bar

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from skimage.feature import local_binary_pattern

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from imblearn.over_sampling import SMOTE
# output: no output if all imports succeed, error if a library is missing

In [ ]:
# CHANGE THIS PATH TO YOUR DATASET FOLDER
BASE_PATH = "cv_dataset/train"

# auto detects class folders inside BASE_PATH — no need to change
class_folders = sorted([d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))])
class_map = {cls: idx for idx, cls in enumerate(class_folders)}

# auto picks first valid image from each class for demos
demo_images = {}
for cls in class_folders:
    folder = os.path.join(BASE_PATH, cls)
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            demo_images[cls] = os.path.join(folder, fname)
            break

first_demo_path = list(demo_images.values())[0]

print("Classes found:", class_map)
print("Demo images:", demo_images)
# output: prints class names like {'apples':0,'tomatoes':1} and demo image paths — verify this looks correct

In [ ]:
def preprocess_image(img):
    img = cv2.resize(img, (128, 128))
    # blur before colour conversion to preserve colour integrity
    img = cv2.medianBlur(img, 5)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    return img, hsv, gray
# output: no output — function is just defined here

In [ ]:
def augment_image(img):
    augmented = [img]
    augmented.append(cv2.flip(img, 1))
    augmented.append(np.clip(img.astype(np.int32) + 30, 0, 255).astype(np.uint8))
    augmented.append(np.clip(img.astype(np.int32) - 30, 0, 255).astype(np.uint8))
    noise = np.random.normal(0, 10, img.shape).astype(np.int32)
    augmented.append(np.clip(img.astype(np.int32) + noise, 0, 255).astype(np.uint8))
    return augmented
# output: no output — function is just defined here

In [ ]:
def show_sample_images(base_path, class_folders, samples=4):
    plt.figure(figsize=(3*samples, 3*len(class_folders)))
    idx = 1
    for cls in class_folders:
        folder = os.path.join(base_path, cls)
        images = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))][:samples]
        for img_name in images:
            img = cv2.imread(os.path.join(folder, img_name))
            if img is None:
                continue
            plt.subplot(len(class_folders), samples, idx)
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.title(cls.capitalize())
            plt.axis("off")
            idx += 1
    plt.suptitle("Sample Images from Dataset", fontsize=14)
    plt.tight_layout()
    plt.show()

show_sample_images(BASE_PATH, class_folders, samples=4)
# output: grid of sample images from each class

In [ ]:
def show_preprocessing_demo(img_path):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img, hsv, gray = preprocess_image(img)
    edges = cv2.Canny(gray, 100, 200)

    plt.figure(figsize=(10, 6))
    plt.subplot(1, 4, 1); plt.imshow(img_rgb); plt.title("Original"); plt.axis("off")
    plt.subplot(1, 4, 2); plt.imshow(gray, cmap="gray"); plt.title("Grayscale + Equalized"); plt.axis("off")
    plt.subplot(1, 4, 3); plt.imshow(hsv[:,:,0], cmap="hsv"); plt.title("HSV (Hue Channel)"); plt.axis("off")
    plt.subplot(1, 4, 4); plt.imshow(edges, cmap="gray"); plt.title("Edge Detection (Canny)"); plt.axis("off")
    plt.tight_layout()
    plt.show()

for cls, path in demo_images.items():
    show_preprocessing_demo(path)
# output: 4-panel preprocessing plot for each class showing original, grayscale, HSV, edges

In [ ]:
def show_augmentation_demo(img_path):
    img_orig = cv2.imread(img_path)
    augs = augment_image(cv2.resize(img_orig, (128, 128)))
    labels = ["Original", "H-Flip", "Brighter", "Darker", "Gaussian Noise"]
    plt.figure(figsize=(16, 3.5))
    for i, (aug, lbl) in enumerate(zip(augs, labels)):
        plt.subplot(1, 5, i+1)
        plt.imshow(cv2.cvtColor(aug, cv2.COLOR_BGR2RGB))
        plt.title(lbl)
        plt.axis('off')
    plt.suptitle("Data Augmentation Techniques", fontsize=14)
    plt.tight_layout()
    plt.show()

show_augmentation_demo(first_demo_path)
# output: 5 augmented versions of one image side by side

In [ ]:
def extract_features(img, hsv, gray):
    features = []

    # colour feature — HSV hue histogram 32 bins
    hist = cv2.calcHist([hsv], [0], None, [32], [0, 180])
    hist = hist.astype("float32")
    hist /= (hist.sum() + 1e-6)
    features.extend(hist.flatten())

    # texture feature — LBP histogram
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    lbp_hist = lbp_hist.astype("float32")
    lbp_hist /= (lbp_hist.sum() + 1e-6)
    features.extend(lbp_hist)

    # shape feature — contour area, perimeter, circularity
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    area = sum(cv2.contourArea(c) for c in contours)
    perimeter = sum(cv2.arcLength(c, True) for c in contours)
    area = np.clip(area, 0, 1e5)
    perimeter = np.clip(perimeter, 0, 1e4)
    circularity = (4 * np.pi * area) / (perimeter**2 + 1e-6)
    features.extend([area, perimeter, circularity])

    return np.array(features, dtype=np.float32)
# output: no output — function is just defined here

In [ ]:
def show_contours(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_img = img.copy()
    cv2.drawContours(contour_img, contours, -1, (0, 255, 0), 2)
    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB))
    plt.title("Contour-Based Shape Features")
    plt.axis("off")
    plt.show()

for cls, path in demo_images.items():
    show_contours(path)
# output: contour image for each class with green outlines drawn on object

In [ ]:
img = cv2.imread(first_demo_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(gray, None)
sift_img = cv2.drawKeypoints(img, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
# output: no output — SIFT keypoints computed and stored

In [ ]:
plt.subplot(3,4,12); plt.imshow(sift_img); plt.title("SIFT"); plt.axis('off')
# output: one image with SIFT keypoints drawn as rich circles

In [ ]:
def load_dataset(base_path, class_map):
    X, y = [], []

    for cls, label in class_map.items():
        folder = os.path.join(base_path, cls)
        num = 0

        for file in os.listdir(folder):
            img_path = os.path.join(folder, file)
            try:
                img = cv2.imread(img_path)

                if img is None:
                    print("Corrupt Image (None):", file)
                    continue

                if img.size == 0:
                    print("Corrupt Image (empty):", file)
                    continue

                if len(img.shape) < 3 or img.shape[2] != 3:
                    print("Corrupt Image (bad shape):", file)
                    continue

                candidates = augment_image(img)

                for candidate in candidates:
                    img_p, hsv, gray = preprocess_image(candidate)
                    features = extract_features(img_p, hsv, gray)
                    X.append(features)
                    y.append(label)

                num += 1
                print("Number of images Done", num, end="\r")

            except Exception as e:
                print("Corrupt Image", e)
                continue

        print(f"\nLabel {label} Done")

    return np.array(X), np.array(y)
# output: no output — function is just defined here

In [ ]:
X, y = load_dataset(BASE_PATH, class_map)

# remove any NaN or Inf values produced by corrupt images
mask = np.isfinite(X).all(axis=1)
X = X[mask]
y = y[mask]

print(f"Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features")
for cls, label in class_map.items():
    print(f"  {cls}: {(y==label).sum()} samples")
# output: prints 'Number of images Done X' per image then 'Label X Done' per class, then total samples

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print(f"After SMOTE — Total: {len(y_resampled)}")
for cls, label in class_map.items():
    print(f"  {cls}: {(y_resampled==label).sum()}")
print(f"Train: {len(y_train)} | Test: {len(y_test)}")
# output: prints class counts after SMOTE then Train/Test sizes

In [ ]:
k_values = [3, 4, 5, 6, 7, 8, 9]
# output: no output — just a list defined

In [ ]:
plt.figure(figsize=(16, 8))

for k in k_values:
    knn_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance", metric="minkowski"))
    ])

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=knn_pipeline,
        X=X_resampled,
        y=y_resampled,
        train_sizes=np.linspace(0.1, 1.0, 6),
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    val_mean = np.mean(val_scores, axis=1)
    plt.plot(train_sizes, val_mean, marker="o", label=f"k = {k}")

plt.xlabel("Training Set Size")
plt.ylabel("Validation Accuracy")
plt.title("Learning Curves of k-NN for Different k Values")
plt.legend()
plt.grid(True)
plt.show()
# output: line chart with one curve per k value showing validation accuracy vs training size

In [ ]:
models = {
    "KNN (k=4)":           KNeighborsClassifier(n_neighbors=4, weights='distance', metric='minkowski'),
    "SVM (RBF)":           SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=10, random_state=42),
    "Naive Bayes":         GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print(f"{'Model':<24} {'Mean CV Acc':>12} {'Std':>8}")
print("-" * 46)

for name, clf in models.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<24} {scores.mean():.4f}       +/-{scores.std():.4f}")
# output: table showing mean CV accuracy and std for all 7 models

In [ ]:
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors = ['#e74c3c' if m == max(means) else '#3498db' for m in means]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names, means, xerr=stds, color=colors, edgecolor='white', height=0.55, capsize=4)
ax.set_xlabel('Mean CV Accuracy', fontsize=12)
ax.set_title('5-Fold CV Model Comparison', fontsize=14)
ax.set_xlim(0.5, 1.02)
for bar, val in zip(bars, means):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)
ax.legend(handles=[mpatches.Patch(color='#e74c3c', label='Best'), mpatches.Patch(color='#3498db', label='Others')], loc='lower right')
plt.tight_layout()
plt.show()

best_name = names[np.argmax(means)]
print(f"Best model: {best_name} (CV Accuracy = {max(means):.4f})")
# output: horizontal bar chart — best model in red, others in blue

In [ ]:
# train knn k=4 first exactly like teacher notebook
k = 4
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance", metric="minkowski"))
])
knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=list(class_map.keys())))
# output: confusion matrix as text numbers then full classification report with precision recall F1

In [ ]:
# train the best model found from CV
best_pipeline = Pipeline([('scaler', StandardScaler()), ('clf', models[best_name])])
best_pipeline.fit(X_train, y_train)
y_pred_best = best_pipeline.predict(X_test)
y_pred_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(f"Best Model ({best_name}) — Test Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=list(class_map.keys())))
# output: test accuracy of best model and its classification report

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(class_map.keys()))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()
# output: blue heatmap confusion matrix — rows actual, columns predicted

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='crimson', lw=2, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.15, color='crimson')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — {best_name}')
ax.legend()
plt.tight_layout()
plt.show()
print(f"AUC-ROC: {roc_auc:.4f}")
# output: ROC curve with AUC score printed below

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_pipeline, X=X_train, y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 8), cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std  = np.std(train_scores,  axis=1)
val_mean   = np.mean(val_scores,   axis=1)
val_std    = np.std(val_scores,    axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, 'o-', color='royalblue', lw=2, label='Training accuracy')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='royalblue')
ax.plot(train_sizes, val_mean, 's-', color='darkorange', lw=2, label='Validation accuracy')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='darkorange')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('Accuracy')
ax.set_title(f'Learning Curve — {best_name}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# output: two curves — blue training accuracy, orange validation accuracy with shaded std bands

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(StandardScaler().fit_transform(X_resampled))

plot_colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22', '#34495e']

fig, ax = plt.subplots(figsize=(7, 5))
for (cls, label), col in zip(class_map.items(), plot_colors):
    mask = y_resampled == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=col, label=cls.capitalize(), alpha=0.5, s=20, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('PCA Feature Space')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# output: 2D scatter plot with each class in different colour showing feature separability

In [ ]:
print(f"{'Model':<24} {'Test Acc':>10}")
print("-" * 36)
for name, clf in models.items():
    p = Pipeline([('s', StandardScaler()), ('c', clf)])
    p.fit(X_train, y_train)
    acc = accuracy_score(y_test, p.predict(X_test))
    marker = " <- BEST" if name == best_name else ""
    print(f"{name:<24} {acc:.4f}{marker}")
# output: table of all 7 models with test accuracy, best model marked with <- BEST

In [ ]:
# ============================================================
# JUPYTER LITE TROUBLESHOOTING — READ IF ANYTHING FAILS
# ============================================================

# PROBLEM 1: 'micropip' not found / await not working
# FIX: delete the micropip cell and instead run this:
# import subprocess; subprocess.run(['pip', 'install', 'imbalanced-learn'])

# PROBLEM 2: 'imbalanced-learn' or 'SMOTE' import fails
# FIX: run this in a new cell first:
# import micropip; await micropip.install('imbalanced-learn')

# PROBLEM 3: cv2 not available
# FIX: run this in a new cell first:
# import micropip; await micropip.install('opencv-python')

# PROBLEM 4: skimage not available
# FIX: run this in a new cell first:
# import micropip; await micropip.install('scikit-image')

# PROBLEM 5: dataset folder not found / FileNotFoundError
# FIX: make sure zip is extracted and folder name matches BASE_PATH exactly
# check your folder name with: import os; print(os.listdir('.'))

# PROBLEM 6: SIFT_create() raises error
# FIX: SIFT needs opencv-contrib, comment out the two SIFT cells and skip them

# PROBLEM 7: n_jobs=-1 causes freeze or error
# FIX: change n_jobs=-1 to n_jobs=1 in all learning_curve and cross_val_score calls

# PROBLEM 8: everything runs but plots dont show
# FIX: add this line at the top of the notebook:
# %matplotlib inline

# output: no output — this is a reference cell only, run only if something breaks